In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_branches = spark.table("digital_banking.silver.silver_branches")

df_dim_branch = df_silver_branches.select(
    # Primary Key
    F.col("branch_id").alias("branch_key"),
    
    # Branch Identification
    F.col("branch_id"),
    F.col("branch_code"),
    F.col("branch_name"),
    
    # Location Hierarchy
    F.col("city"),
    F.col("state"),
    F.col("region"),
    F.concat_ws(", ", F.col("city"), F.col("state")).alias("city_state"),
    F.concat_ws(", ", F.col("branch_name"), F.col("city"), F.col("state")).alias("full_branch_location"),
    
    # Branch Classification
    F.coalesce(F.col("branch_type"), F.lit("Standard")).alias("branch_type"),
    F.coalesce(F.col("branch_status"), F.lit("Unknown")).alias("branch_status"),
    
    # Branch Type Category
    F.when(F.col("branch_type").isin(["Main", "Regional"]), "Core Branch")
     .when(F.col("branch_type").isin(["Sub", "Satellite"]), "Sub Branch")
     .when(F.col("branch_type").isin(["ATM", "Kiosk"]), "Service Point")
     .otherwise("Standard").alias("branch_category"),
    
    # Management Information
    F.col("manager_name"),
    F.when(F.col("manager_name").isNotNull(), True)
     .otherwise(False).alias("has_manager_assigned"),
    
    # Status Indicators
    F.when(F.col("branch_status") == "Operational", True)
     .otherwise(False).alias("is_operational"),
    F.when(F.col("branch_status").isin(["Operational", "Active"]), True)
     .otherwise(False).alias("is_active"),
    
    # Temporal Attributes
    F.col("opening_date"),
    
    # Branch Age Metrics
    F.datediff(F.current_date(), F.col("opening_date")).alias("days_since_opening"),
    F.floor(F.datediff(F.current_date(), F.col("opening_date")) / 365.25).alias("branch_age_years"),
    
    # Branch Age Group
    F.when(F.datediff(F.current_date(), F.col("opening_date")) < 365, "Less than 1 year")
     .when(F.datediff(F.current_date(), F.col("opening_date")) < 1825, "1-5 years")
     .when(F.datediff(F.current_date(), F.col("opening_date")) < 3650, "5-10 years")
     .when(F.datediff(F.current_date(), F.col("opening_date")) < 7300, "10-20 years")
     .otherwise("20+ years").alias("branch_age_group"),
    
    # Data Quality Flags (from silver layer validation)
    F.coalesce(F.col("is_valid_branch_id"), F.lit(False)).alias("is_valid_branch_id"),
    F.coalesce(F.col("is_valid_branch_code"), F.lit(False)).alias("is_valid_branch_code"),
    F.coalesce(F.col("is_location_complete"), F.lit(False)).alias("is_location_complete"),
    F.coalesce(F.col("is_valid_status"), F.lit(False)).alias("is_valid_status"),
    
    # Overall Data Quality Score
    (
        F.when(F.col("is_valid_branch_id"), 1).otherwise(0) +
        F.when(F.col("is_valid_branch_code"), 1).otherwise(0) +
        F.when(F.col("is_location_complete"), 1).otherwise(0) +
        F.when(F.col("is_valid_status"), 1).otherwise(0)
    ).alias("data_quality_score"),
    
    # Audit Columns
    F.current_timestamp().alias("dimension_created_at"),
    F.current_timestamp().alias("dimension_updated_at")
)

# Write to gold layer as a managed Delta table
df_dim_branch.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.gold.dim_branch")

print(f"Total branches: {df_dim_branch.count()}")

# Display sample
display(df_dim_branch.limit(10))